# Automated Colab evaluation

Chọn **Runtime → Run all** để chạy toàn bộ pipeline. Các cell được tách theo từng giai đoạn để dễ theo dõi, chạy lại và tìm lỗi.

> Google Colab vẫn yêu cầu xác nhận quyền truy cập Drive/Sheets; đây là bước bảo mật không thể tự động bỏ qua.


In [ ]:
#@title 1. Cấu hình chạy
REPO_URL = "https://github.com/doantrunghieu08/optimization_model_monocular_3.git"  #@param {type:"string"}
REPO_BRANCH = "version_raycasting"  #@param {type:"string"}
DATA_ROOT = "/content/drive/MyDrive/belief_driven_occlusion_detection_and_error_correction_4_3DHPE_using_uncalibrated_dual_cams"  #@param {type:"string"}
SUBJECTS = "S8"  #@param {type:"string"}
SEQUENCES = "Seq1"  #@param {type:"string"}
SEGMENTS = "*"  #@param {type:"string"}
EXCLUDED_CAMERAS = "1"  #@param {type:"string"}
ALPHA = 0.1  #@param {type:"number"}
BETA = 0.85  #@param {type:"number"}
ENABLE_LEARNABLE = True  #@param {type:"boolean"}
ENABLE_LEARNABLE_EXTRA = True  #@param {type:"boolean"}
SPREADSHEET_NAME = ""  #@param {type:"string"}
SMPL_NEUTRAL_FILE_ID = "1xblXsbK1rTSn5cG934cDhRFB0Apn64Ll"  #@param {type:"string"}
AUTO_CONFIG_FROM_FILENAME = True  #@param {type:"boolean"}

In [ ]:
#@title 2. Chuẩn bị repository và môi trường
import importlib
import json
import os
import re
import socket
import shutil
import subprocess
import sys
from pathlib import Path
from urllib.request import urlopen


def current_notebook_name():
    try:
        host = socket.gethostbyname(socket.gethostname())
        with urlopen(f"http://{host}:9000/api/sessions", timeout=3) as response:
            sessions = json.load(response)
        return sessions[0].get("name", "") if sessions else ""
    except Exception:
        return ""


if AUTO_CONFIG_FROM_FILENAME:
    notebook_name = current_notebook_name()
    match = re.search(r"_alpha(\d+(?:\.\d+)?[Ee][_-]?\d+)_beta(\d+(?:\.\d+)?[Ee][_-]?\d+)", notebook_name)
    if match:
        ALPHA = float(match.group(1).replace("E_", "E-").replace("e_", "e-"))
        BETA = float(match.group(2).replace("E_", "E-").replace("e_", "e-"))
        print(f"Read alpha={ALPHA}, beta={BETA} from {notebook_name}")
    os.environ["LOCAL_METHOD"] = "naive_distance_belief" if "_naive_" in notebook_name else "optical_aware_belief"
    os.environ["KINEMATIC_CONSTRAINTS"] = str("_kinematic_" in notebook_name)
    os.environ["GLOBAL"] = str("_local_" not in notebook_name)
    os.environ["LOSS_TYPE"] = "mse" if "_mse_" in notebook_name else "huber"
    version_match = re.search(r"_(\d{6})_", notebook_name)
    if version_match:
        os.environ["CODE_VERSION"] = version_match.group(1)


def run(command, cwd=None):
    print("$", " ".join(map(str, command)))
    subprocess.run(command, cwd=cwd, check=True)


repo_dir = Path("/content/optimization_model_monocular_3")
if (repo_dir / ".git").exists():
    if (repo_dir / ".git").exists():
      print(f"Updating branch: {REPO_BRANCH}")

    # Fetch branch và tạo/cập nhật chính xác remote-tracking ref
    run([
        "git",
        "fetch",
        "origin",
        f"{REPO_BRANCH}:refs/remotes/origin/{REPO_BRANCH}"
    ], repo_dir)

    # Tạo/reset local branch theo remote
    run([
        "git",
        "checkout",
        "-f",
        "-B",
        REPO_BRANCH,
        f"origin/{REPO_BRANCH}"
    ], repo_dir)
elif repo_dir.exists():
    backup_dir = repo_dir.with_name(f"{repo_dir.name}_backup_{os.getpid()}")
    repo_dir.rename(backup_dir)
    print(f"Moved the existing non-Git directory to {backup_dir}")
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)])
else:
    run(["git", "clone", "--depth", "1", "--branch", REPO_BRANCH, REPO_URL, str(repo_dir)])

# Install everything before importing project/Colab dependencies.
requirements_path = repo_dir / "requirements.txt"
if sys.version_info >= (3, 13):
    requirements_lines = requirements_path.read_text(encoding="utf-8").splitlines()
    requirements_lines = [
        line for line in requirements_lines
        if not re.match(r"^\s*(numpy|scipy)(?:[<>=!~;]|$)", line, re.IGNORECASE)
    ]
    requirements_lines.extend(["numpy>=2.1,<2.3", "scipy>=1.14.1,<1.15"])
    requirements_path = Path("/tmp/requirements-colab.txt")
    requirements_path.write_text("\n".join(requirements_lines) + "\n", encoding="utf-8")

run([sys.executable, "-m", "pip", "install", "-q", "-r", str(requirements_path)])
run([
    sys.executable, "-m", "pip", "install", "-q",
    "gspread", "google-api-python-client", "pandas", "gdown", "PyYAML",
])

scientific_check = [
    sys.executable,
    "-c",
    "import numpy, scipy; from scipy.spatial.distance import cdist; "
    "print('NumPy', numpy.__version__, '| SciPy', scipy.__version__)",
]
check_result = subprocess.run(scientific_check, text=True, capture_output=True)
if check_result.returncode:
    print("Repairing the incompatible NumPy/SciPy installation...")
    scientific_specs = (
        ["numpy>=2.1,<2.3", "scipy>=1.14.1,<1.15"]
        if sys.version_info >= (3, 13)
        else ["numpy<2", "scipy<2"]
    )
    run([
        sys.executable, "-m", "pip", "install", "-q", "--upgrade",
        "--force-reinstall", "--no-cache-dir", *scientific_specs,
    ])
    run(scientific_check)
else:
    print(check_result.stdout.strip())

if shutil.which("ffmpeg") is None:
    run(["apt-get", "update", "-qq"])
    run(["apt-get", "install", "-y", "-qq", "ffmpeg"])
importlib.invalidate_caches()


Read alpha=0.1, beta=0.85 from ablation_hieuDT_belief_fusion_H260912RayCasting_naive_kinematic_alpha1E_1_beta85E_2.ipynb
Updating branch: version_raycasting
$ git fetch origin version_raycasting:refs/remotes/origin/version_raycasting
$ git checkout -f -B version_raycasting origin/version_raycasting
$ /usr/bin/python3 -m pip install -q -r /tmp/requirements-colab.txt
$ /usr/bin/python3 -m pip install -q gspread google-api-python-client pandas gdown PyYAML
NumPy 2.1.3 | SciPy 1.14.1


In [ ]:
#@title 3. Kết nối Google Drive và tải model
from google.colab import auth, drive
from google.auth import default
from googleapiclient.discovery import build
import gdown
import yaml
from ruamel.yaml import YAML

drive.mount("/content/drive", force_remount=False)
auth.authenticate_user()
credentials, _ = default()

try:
    user = build("drive", "v3", credentials=credentials).about().get(fields="user").execute()["user"]
    os.environ["RUNNER_NAME"] = user.get("displayName", "Colab_User")
    os.environ["RUNNER_EMAIL"] = user.get("emailAddress", "")
except Exception as exc:
    print(f"Could not read Drive profile ({exc}); using Colab_User.")
    os.environ["RUNNER_NAME"] = "Colab_User"

model_path = repo_dir / "models" / "SMPL_NEUTRAL.pkl"
model_path.parent.mkdir(parents=True, exist_ok=True)
if not model_path.exists() or model_path.stat().st_size < 1_000_000:
    print("Downloading SMPL_NEUTRAL.pkl...")
    downloaded = gdown.download(id=SMPL_NEUTRAL_FILE_ID, output=str(model_path), quiet=False)
    if not downloaded or not model_path.exists():
        raise RuntimeError("Could not download SMPL_NEUTRAL.pkl. Check the Drive file ID/access permission.")


Mounted at /content/drive


Downloading...
From: https://drive.google.com/uc?id=1xblXsbK1rTSn5cG934cDhRFB0Apn64Ll
To: /content/optimization_model_monocular_3/models/SMPL_NEUTRAL.pkl
100%|██████████| 39.0M/39.0M [00:01<00:00, 37.1MB/s]


In [ ]:
#@title 4. Áp dụng cấu hình pipeline
# Các tham số fusion (ALPHA, BETA, LOCAL_METHOD, GLOBAL, KINEMATIC_CONSTRAINTS, LOSS_TYPE)
# được tự động đọc từ tên notebook bởi set_env_from_filename() trong run_brute_force().
# Chỉ cần set các flag learnable ở đây vì chúng không được encode trong tên file.

import os
os.environ["ENABLE_LEARNABLE"]       = str(ENABLE_LEARNABLE).lower()
os.environ["ENABLE_LEARNABLE_EXTRA"] = str(ENABLE_LEARNABLE_EXTRA).lower()

# Patch pipeline.yml: chỉ cập nhật learnable flags, không đụng đến fusion params
from pathlib import Path
from ruamel.yaml import YAML

pipeline_path = repo_dir / "configs" / "pipeline.yml"
roundtrip_yaml = YAML()
with pipeline_path.open("r", encoding="utf-8") as stream:
    pipeline_config = roundtrip_yaml.load(stream)
pipeline_config["learnable"]["enabled"]       = bool(ENABLE_LEARNABLE)
pipeline_config["learnable_extra"]["enabled"] = bool(ENABLE_LEARNABLE_EXTRA)
with pipeline_path.open("w", encoding="utf-8") as stream:
    roundtrip_yaml.dump(pipeline_config, stream)

keypoint_map_path = repo_dir / "configs" / "keypoints3D_map.yml"
with keypoint_map_path.open("r", encoding="utf-8") as stream:
    keypoint_map = roundtrip_yaml.load(stream)
all_joint_names = [item["name"] for item in keypoint_map["keypoints"]]
if len(keypoint_map.get("priority2", [])) != len(all_joint_names):
    keypoint_map["priority2"] = all_joint_names
    with keypoint_map_path.open("w", encoding="utf-8") as stream:
        roundtrip_yaml.dump(keypoint_map, stream)

In [ ]:
#@title 5. Dò dữ liệu và tạo brute_force.yml
def accepts(filter_text, value):
    requested = {item.strip() for item in filter_text.split(",") if item.strip()}
    return not requested or "*" in requested or value in requested


def find_video(image_sequence, segments_dir, pkl_path, camera_id, segment_id):
    candidates = [
        pkl_path.parent / "output.mp4",
        pkl_path.parent / f"video_{camera_id}_seg_{segment_id}.mp4",
        segments_dir / f"video_{camera_id}_seg_{segment_id}.mp4",
        image_sequence / f"video_{camera_id}.avi",
        image_sequence / f"video_{camera_id}.mp4",
    ]
    return next((path for path in candidates if path.exists()), None)


data_root = Path(DATA_ROOT).expanduser()
if not data_root.exists():
    raise FileNotFoundError(f"DATA_ROOT does not exist: {data_root}")

image_sequences = [data_root] if data_root.name == "imageSequence" else sorted(data_root.rglob("imageSequence"))
excluded_cameras = {item.strip() for item in EXCLUDED_CAMERAS.split(",") if item.strip()}
discovered_segments = []

for image_sequence in image_sequences:
    sequence = image_sequence.parent.name
    subject = image_sequence.parent.parent.name
    if not accepts(SUBJECTS, subject) or not accepts(SEQUENCES, sequence):
        continue

    segments_dir = next((image_sequence / name for name in ("Segments", "segments") if (image_sequence / name).is_dir()), None)
    gt_dir = next((path for path in (image_sequence / "GT", image_sequence.parent / "GT") if path.is_dir()), None)
    if segments_dir is None or gt_dir is None:
        print(f"Skipping {subject}/{sequence}: missing Segments or GT directory.")
        continue

    grouped = {}
    for pkl_path in sorted(segments_dir.rglob("*.pkl")):
        match = re.search(r"video_(\d+)_seg_(\d+)$", pkl_path.stem)
        if not match:
            continue
        camera_id, segment_id = match.groups()
        segment_name = f"seg_{segment_id}"
        if camera_id in excluded_cameras or not accepts(SEGMENTS, segment_name):
            continue
        if not (gt_dir / f"video_{camera_id}_{segment_name}.json").exists():
            continue
        video_path = find_video(image_sequence, segments_dir, pkl_path, camera_id, segment_id)
        if video_path is None:
            continue
        grouped.setdefault(segment_name, {})[camera_id] = {
            "id": f"video_{camera_id}_{segment_name}",
            "pkl": str(pkl_path.resolve()),
            "video": str(video_path.resolve()),
        }

    for segment_name, cameras_by_id in sorted(grouped.items()):
        cameras = [cameras_by_id[key] for key in sorted(cameras_by_id, key=int)]
        if len(cameras) >= 2:
            discovered_segments.append({
                "name": f"{subject}_{sequence}_{segment_name}",
                "ground_truth_dir": str(gt_dir.resolve()),
                "cameras": cameras,
            })

if not discovered_segments:
    raise RuntimeError(
        "No runnable segment with at least two cameras was found. "
        "Expected PKL names like video_0_seg_1.pkl and matching GT JSON files."
    )

brute_force_path = repo_dir / "configs" / "brute_force.yml"
with brute_force_path.open("w", encoding="utf-8") as stream:
    yaml.safe_dump({"segments": discovered_segments}, stream, sort_keys=False, allow_unicode=True)

camera_count = sum(len(segment["cameras"]) for segment in discovered_segments)
pair_count = sum(len(segment["cameras"]) * (len(segment["cameras"]) - 1) for segment in discovered_segments)
print(f"Discovered {len(discovered_segments)} segments, {camera_count} camera inputs, {pair_count} ordered pairs.")
print(f"Configuration: alpha={ALPHA}, beta={BETA}, learnable={ENABLE_LEARNABLE}, learnable_extra={ENABLE_LEARNABLE_EXTRA}")


Discovered 7 segments, 42 camera inputs, 210 ordered pairs.
Configuration: alpha=0.1, beta=0.85, learnable=True, learnable_extra=True


In [ ]:
#@title 6. Chạy đánh giá
os.chdir(repo_dir)
sys.path.insert(0, str(repo_dir))
import brute_force_runner

# Avoid the runner's timed input: blank means its deterministic per-user default.
brute_force_runner.get_spreadsheet_name_input = (
    lambda default_name, timeout=10: SPREADSHEET_NAME.strip() or default_name
)
brute_force_runner.run_brute_force()
print("✅ Finished. The final Google Sheets URL is printed above.")


/content/optimization_model_monocular_3/config_loader.py:226: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'object'):
/content/optimization_model_monocular_3/config_loader.py:234: FutureWarning: In the future `np.str` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, 'str'):


Learnable backend ready: /content/optimization_model_monocular_3/_learnable_backend
 - files: 12
 - src: /content/optimization_model_monocular_3/_learnable_backend/src
 - config: /content/optimization_model_monocular_3/_learnable_backend/src/config/net.yaml
Alpha trong config: 0.1
Beta trong config: 0.85

=== Bắt đầu vét cạn cho Segment: S8_Seq1_seg_1 (30 cặp) ===

--- [Tiến trình: 1/210] Master=video_0_seg_1 | Supplement=video_2_seg_1 ---
[Preprocess] Start | output=/content/optimization_model_monocular_3/output/preprocess_results
[Preprocess] Offset input | cam1=video_0_seg_1.pkl | cam2=video_2_seg_1.pkl
[Preprocess] Offset=0
[Preprocess] 2D skip: tracking_results_for_reproj missing in /content/drive/.shortcut-targets-by-id/13k5qCwFEImOMdOHOUazTj0ruuPmRBeWb/belief_driven_occlusion_detection_and_error_correction_4_3DHPE_using_uncalibrated_dual_cams/S8/Seq1/imageSequence/Segments/wham_output_video_0_seg_1/video_0_seg_1.pkl
[Preprocess] 2D skip: tracking_results_for_reproj missing in /c

[Pose] Exporting JSONs: 100%|██████████| 424/424 [00:04<00:00, 91.77frame/s]
Traceback (most recent call last):
  File "/content/optimization_model_monocular_3/brute_force_runner.py", line 435, in _evaluate_camera_pair
    run_pipeline(config, stage_override=None)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/optimization_model_monocular_3/pipeline.py", line 59, in run_pipeline
    _run_full_pipeline(config, include_visualization=False)
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/optimization_model_monocular_3/pipeline.py", line 38, in _run_full_pipeline
    run_fusion(config)
    ~~~~~~~~~~^^^^^^^^
  File "/content/optimization_model_monocular_3/fusion_pipeline/executor.py", line 300, in run_fusion
    torso_faces = load_torso_faces(paths["segmentation"], faces) if mesh_loaded else None
                  ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/optimization_model_monocular_3/fusion_pipeline/detector.py", line 58,

[Pose] Done. Output: /content/optimization_model_monocular_3/output/pose_results
[Pipeline] Running fusion with offset=0
[Fusion] Camera-space mesh cache: 424 synced frames
Lỗi khi chạy cặp video_0_seg_1-video_2_seg_1: Segmentation mask file not found: /content/optimization_model_monocular_3/models/smpl_partSegmentation_mapping.pkl

--- [Tiến trình: 2/210] Master=video_0_seg_1 | Supplement=video_4_seg_1 ---
[Preprocess] Start | output=/content/optimization_model_monocular_3/output/preprocess_results
[Preprocess] Offset input | cam1=video_0_seg_1.pkl | cam2=video_4_seg_1.pkl
[Preprocess] Offset=0
[Preprocess] 2D skip: tracking_results_for_reproj missing in /content/drive/.shortcut-targets-by-id/13k5qCwFEImOMdOHOUazTj0ruuPmRBeWb/belief_driven_occlusion_detection_and_error_correction_4_3DHPE_using_uncalibrated_dual_cams/S8/Seq1/imageSequence/Segments/wham_output_video_0_seg_1/video_0_seg_1.pkl
[Preprocess] 2D skip: tracking_results_for_reproj missing in /content/drive/.shortcut-targets-by

[Pose] Exporting JSONs: 100%|██████████| 424/424 [00:04<00:00, 87.38frame/s]
Traceback (most recent call last):
  File "/content/optimization_model_monocular_3/brute_force_runner.py", line 435, in _evaluate_camera_pair
    run_pipeline(config, stage_override=None)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/optimization_model_monocular_3/pipeline.py", line 59, in run_pipeline
    _run_full_pipeline(config, include_visualization=False)
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/optimization_model_monocular_3/pipeline.py", line 38, in _run_full_pipeline
    run_fusion(config)
    ~~~~~~~~~~^^^^^^^^
  File "/content/optimization_model_monocular_3/fusion_pipeline/executor.py", line 300, in run_fusion
    torso_faces = load_torso_faces(paths["segmentation"], faces) if mesh_loaded else None
                  ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/optimization_model_monocular_3/fusion_pipeline/detector.py", line 58,

...

[Pose] Exporting JSONs: 100%|██████████| 781/781 [00:08<00:00, 94.40frame/s]


[Pose] Done. Output: /content/optimization_model_monocular_3/output/pose_results
[Pipeline] Running fusion with offset=0
[Fusion] Camera-space mesh cache: 781 synced frames
Lỗi khi chạy cặp video_2_seg_5-video_8_seg_5: Segmentation mask file not found: /content/optimization_model_monocular_3/models/smpl_partSegmentation_mapping.pkl

--- [Tiến trình: 101/210] Master=video_4_seg_5 | Supplement=video_0_seg_5 ---
[Preprocess] Start | output=/content/optimization_model_monocular_3/output/preprocess_results
[Preprocess] Offset input | cam1=video_4_seg_5.pkl | cam2=video_0_seg_5.pkl


Traceback (most recent call last):
  File "/content/optimization_model_monocular_3/brute_force_runner.py", line 435, in _evaluate_camera_pair
    run_pipeline(config, stage_override=None)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/optimization_model_monocular_3/pipeline.py", line 59, in run_pipeline
    _run_full_pipeline(config, include_visualization=False)
    ~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/optimization_model_monocular_3/pipeline.py", line 38, in _run_full_pipeline
    run_fusion(config)
    ~~~~~~~~~~^^^^^^^^
  File "/content/optimization_model_monocular_3/fusion_pipeline/executor.py", line 300, in run_fusion
    torso_faces = load_torso_faces(paths["segmentation"], faces) if mesh_loaded else None
                  ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/optimization_model_monocular_3/fusion_pipeline/detector.py", line 58, in load_torso_faces
    raise FileNotFoundError(f"Segmentation mask file not

[Preprocess] Offset=-1
[Preprocess] 2D skip: tracking_results_for_reproj missing in /content/drive/.shortcut-targets-by-id/13k5qCwFEImOMdOHOUazTj0ruuPmRBeWb/belief_driven_occlusion_detection_and_error_correction_4_3DHPE_using_uncalibrated_dual_cams/S8/Seq1/imageSequence/Segments/wham_output_video_4_seg_5/video_4_seg_5.pkl
[Preprocess] 2D skip: tracking_results_for_reproj missing in /content/drive/.shortcut-targets-by-id/13k5qCwFEImOMdOHOUazTj0ruuPmRBeWb/belief_driven_occlusion_detection_and_error_correction_4_3DHPE_using_uncalibrated_dual_cams/S8/Seq1/imageSequence/Segments/wham_output_video_0_seg_5/video_0_seg_5.pkl
[Preprocess] Selected offset | offset=-1 | file=data_cam1.json
[Preprocess] Done | output=/content/optimization_model_monocular_3/output/preprocess_results
[Pipeline] Running pose with offset=-1
[Pose] Loading SMPL model...
[Pose] Loading camera 1: /content/drive/.shortcut-targets-by-id/13k5qCwFEImOMdOHOUazTj0ruuPmRBeWb/belief_driven_occlusion_detection_and_error_correctio

[Pose] Exporting JSONs:  46%|████▌     | 238/521 [00:03<00:02, 95.11frame/s]